# **Comparative Implementation of ML Regression Algorithms**
**Dataset:** California Housing (scikit-learn) — predict median house value of a district.

**Algorithms:** Linear Regression, Decision Tree, Random Forest, Neural Network (MLP), XGBoost, K-Nearest Neighbors.

**Fair-comparison rules:** same 8 input features, same target (`MedHouseVal`, in units of $100,000), same 80:20 train/test split (`random_state=42`), same metrics (MAE, MSE, RMSE, R²).

In [1]:
# If running in Colab and xgboost is missing: !pip install xgboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

import os
os.makedirs("images", exist_ok=True)

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

## 1. Load and explore the dataset

In [2]:
data = fetch_california_housing(as_frame=True)
df = data.frame
print("Shape:", df.shape)
df.head()

Shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
print("Missing values per column:\n", df.isnull().sum())
df.describe().T.round(3)

Missing values per column:
 MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.871,1.900,0.500,2.563,3.535,4.743,15.000
HouseAge,20640.0,28.639,12.586,1.000,18.000,29.000,37.000,52.000
AveRooms,20640.0,5.429,2.474,0.846,4.441,5.229,6.052,141.909
AveBedrms,20640.0,1.097,0.474,0.333,1.006,1.049,1.100,34.067
Population,20640.0,1425.477,1132.462,3.000,787.000,1166.000,1725.000,35682.000
AveOccup,20640.0,3.071,10.386,0.692,2.430,2.818,3.282,1243.333
Latitude,20640.0,35.632,2.136,32.540,33.930,34.260,37.710,41.950
Longitude,20640.0,-119.570,2.004,-124.350,-121.800,-118.490,-118.010,-114.310
MedHouseVal,20640.0,2.069,1.154,0.150,1.196,1.797,2.647,5.000


**Features (8):** `MedInc` (median income, $10k), `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`.
**Target:** `MedHouseVal` (median house value, $100k). The dataset has no missing values. Feature ranges differ hugely (e.g. `Population` up to ~35,000 vs `AveBedrms` ~1), so distance- and gradient-based models need scaling.

In [4]:
corr = df.corr()["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)
print("Correlation of each feature with the target:\n", corr.round(3))

Correlation of each feature with the target:
 MedInc        0.688
AveRooms      0.152
HouseAge      0.106
AveOccup     -0.024
Population   -0.025
Longitude    -0.046
AveBedrms    -0.047
Latitude     -0.144
Name: MedHouseVal, dtype: float64


## 2. Preprocessing: features, target, 80:20 split, scaling

In [5]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
print("Train:", X_train.shape, " Test:", X_test.shape)

# Scaler is fitted on TRAINING data only (no data leakage), then applied to test data
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

Train: (16512, 8)  Test: (4128, 8)


**Scaling policy:** Linear Regression, Neural Network and KNN use the standardised features (z = (x − μ)/σ). Tree-based models (Decision Tree, Random Forest, XGBoost) split on thresholds, so scaling does not change them; they use the raw features. Every model still sees the same 8 features and the same rows.

## 3. Define the six models

In [6]:
models = {
    "Linear Regression": (LinearRegression(), True),
    "Decision Tree":     (DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=SEED), False),
    "Random Forest":     (RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=SEED), False),
    "Neural Network":    (MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
                                       learning_rate_init=0.001, max_iter=500, early_stopping=True,
                                       random_state=SEED), True),
    "XGBoost":           (XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                                       colsample_bytree=0.8, random_state=SEED, n_jobs=-1), False),
    "KNN":               (KNeighborsRegressor(n_neighbors=7, weights="distance"), True),
}
# The boolean says whether the model uses scaled features

### Choosing k for KNN (5-fold CV on the training set only)

In [7]:
ks = [1, 3, 5, 7, 9, 11, 15, 21]
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
k_scores = [cross_val_score(KNeighborsRegressor(n_neighbors=k, weights="distance"),
                            X_train_s, y_train, cv=cv, scoring="r2").mean() for k in ks]
for k, s in zip(ks, k_scores):
    print(f"k={k:>2}  mean CV R2 = {s:.4f}")
best_k = ks[int(np.argmax(k_scores))]
print("Best k:", best_k)
models["KNN"] = (KNeighborsRegressor(n_neighbors=best_k, weights="distance"), True)

k= 1  mean CV R2 = 0.5173
k= 3  mean CV R2 = 0.6624
k= 5  mean CV R2 = 0.6856
k= 7  mean CV R2 = 0.6914
k= 9  mean CV R2 = 0.6951
k=11  mean CV R2 = 0.6961
k=15  mean CV R2 = 0.6941
k=21  mean CV R2 = 0.6913
Best k: 11


## 4. Train and evaluate every model on the same test set

In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}

results, preds, fitted = [], {}, {}
for name, (model, needs_scaling) in models.items():
    Xtr, Xte = (X_train_s, X_test_s) if needs_scaling else (X_train, X_test)
    t0 = time.time()
    model.fit(Xtr, y_train)
    train_time = time.time() - t0
    y_pred = model.predict(Xte)
    preds[name], fitted[name] = y_pred, model
    row = {"Model": name, **evaluate(y_test, y_pred),
           "Train R2": r2_score(y_train, model.predict(Xtr)), "Train time (s)": train_time}
    results.append(row)
    print(f"{name:<18} R2={row['R2']:.4f}  RMSE={row['RMSE']:.4f}")

Linear Regression  R2=0.5758  RMSE=0.7456
Decision Tree      R2=0.6889  RMSE=0.6385


## 5. Comparative performance table

In [ ]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1
results_df.round(4)

In [ ]:
print("Linear Regression coefficients (on standardised features):")
lr = fitted["Linear Regression"]
print(pd.Series(lr.coef_, index=X.columns).round(4).to_string())
print("Intercept:", round(lr.intercept_, 4))

### Robustness check: 5-fold cross-validation R² on the training set

In [ ]:
cv_rows = []
for name, (model, needs_scaling) in models.items():
    est = make_pipeline(StandardScaler(), model) if needs_scaling else model
    s = cross_val_score(est, X_train, y_train, cv=cv, scoring="r2", n_jobs=1)
    cv_rows.append({"Model": name, "CV R2 mean": s.mean(), "CV R2 std": s.std()})
cv_df = pd.DataFrame(cv_rows).sort_values("CV R2 mean", ascending=False).reset_index(drop=True)
cv_df.round(4)

## 6. Visualisation: actual vs predicted

In [ ]:
order = results_df["Model"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5), sharex=True, sharey=True)
for ax, name in zip(axes.ravel(), order):
    p = preds[name]
    r = results_df.set_index("Model").loc[name]
    ax.scatter(y_test, p, s=8, alpha=0.25, color="#2a78d6", edgecolors="none")
    ax.plot([0, 5.2], [0, 5.2], color="#eb6834", lw=2, label="Perfect prediction (y = x)")
    ax.set_title(f"{name}\nR² = {r['R2']:.3f}   RMSE = {r['RMSE']:.3f}", fontsize=11)
    ax.set_xlim(0, 5.3); ax.set_ylim(-0.5, 6)
for ax in axes[-1]: ax.set_xlabel("Actual value ($100k)")
for ax in axes[:, 0]: ax.set_ylabel("Predicted value ($100k)")
axes[0, 0].legend(loc="upper left", frameon=False, fontsize=9)
fig.suptitle("Actual vs predicted house value on the test set (4,128 districts)", fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig("images/actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
names = results_df["Model"][::-1]
for ax, metric in zip(axes, ["R2", "RMSE", "MAE"]):
    vals = results_df.set_index("Model").loc[names, metric]
    ax.barh(names, vals, color="#2a78d6", height=0.6)
    for i, v in enumerate(vals):
        ax.text(v, i, f" {v:.3f}", va="center", fontsize=9)
    ax.set_title(metric + (" (higher is better)" if metric == "R2" else " (lower is better)"))
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, vals.max() * 1.18)
plt.tight_layout()
plt.savefig("images/metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Line plot of the first 60 test samples: actual vs the best and worst models
best, worst = order[0], order[-1]
idx = np.arange(60)
plt.figure(figsize=(14, 4.5))
plt.plot(idx, y_test.values[:60], color="#1f1f1f", lw=2, marker="o", ms=4, label="Actual")
plt.plot(idx, preds[best][:60], color="#2a78d6", lw=2, marker="s", ms=4, label=f"{best} (best)")
plt.plot(idx, preds[worst][:60], color="#eb6834", lw=2, ls="--", label=f"{worst} (lowest R²)")
plt.xlabel("Test sample index"); plt.ylabel("House value ($100k)")
plt.title("Actual vs predicted for the first 60 test districts")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("images/sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Feature importance from the best tree ensemble
imp = pd.Series(fitted["XGBoost"].feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(imp.index, imp.values, color="#2a78d6", height=0.6)
plt.title("XGBoost feature importance (gain-based)")
plt.grid(axis="y", visible=False)
plt.tight_layout()
plt.savefig("images/xgb_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Best model

In [ ]:
best_row = results_df.iloc[0]
print(f"Best model: {best_row['Model']}")
print(f"  R2   = {best_row['R2']:.4f}  (explains {best_row['R2']*100:.1f}% of variance in house value)")
print(f"  RMSE = {best_row['RMSE']:.4f}  (≈ ${best_row['RMSE']*100000:,.0f} typical error)")
print(f"  MAE  = {best_row['MAE']:.4f}  (≈ ${best_row['MAE']*100000:,.0f} average absolute error)")